# Комбинация паттернов: Supervisor + Reflection — «Виртуальная редакция»

Это пример архитектуры, близкой к реальному продукту. Автоматическая редакция исследует тему, пишет статью и доводит её до публикабельного качества. Супервайзер координирует команду из трёх агентов, а внутри пары «писатель — редактор» крутится невидимый ему цикл рефлексии.

Супервайзер маршрутизирует задачу: сначала исследователь собирает факты, затем писатель и редактор работают в петле рефлексии. Супервайзер не видит внутренний цикл — он отправляет задачу «написать текст» и получает отполированный результат. Принцип матрёшки в действии.

```mermaid
graph LR
    START((START)) --> supervisor{{"🎯 Supervisor<br/><i>маршрутизирует</i>"}}
    supervisor -->|researcher| researcher(["🔹 Researcher<br/><i>ищет факты</i>"])
    supervisor -->|writer| writer(["🔹 Writer<br/><i>пишет черновик</i>"])
    supervisor -->|FINISH| END((END))
    researcher --> supervisor
    writer --> editor(["📊 Editor<br/><i>оценивает качество</i>"])
    editor -->|замечания| writer
    editor -->|APPROVED| supervisor

    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef aggregator fill:#F59E0B,stroke:#D97706,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff

    class START,END terminal
    class supervisor coordinator
    class researcher,writer worker
    class editor aggregator
```

In [1]:
import sys
sys.path.insert(0, "..")

import json
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Состояние редакции

Состояние `EditorialState` содержит семь полей. Ключевое — `iteration`: оно считает циклы Writer-Editor. Супервайзер его не трогает — это внутреннее дело подграфа рефлексии. Поле `next_agent` позволяет супервайзеру и редактору сообщать маршрутизатору, куда направить управление дальше.

In [3]:
llm = get_llm()

class EditorialState(TypedDict):
    task: str              # Задача от пользователя
    research: str          # Результаты исследования
    draft: str             # Текущий черновик
    feedback: str          # Замечания редактора
    final_text: str        # Финальный текст
    next_agent: str        # Кого вызвать следующим
    iteration: int         # Счётчик итераций рефлексии Writer↔Editor

## Инструменты агентов

Каждый агент в редакции имеет свой набор инструментов. Исследователь использует `search_web` для поиска фактов, писатель — `format_article` для оформления текста по стандарту, редактор — `evaluate_text` для объективной оценки качества. В реальной системе здесь были бы настоящие API (Tavily, Serper и т.д.), но для демонстрации принципа достаточно заглушек.

In [4]:
@tool
def search_web(query: str) -> str:
    """Поиск информации в интернете по запросу."""
    # Имитация поискового API — в реальности здесь Tavily, Serper и т.д.
    knowledge = {
        "искусственный интеллект": (
            "ИИ переживает бум: GPT-5.4, Claude 4, Gemini 2.5 демонстрируют "
            "способности к сложному рассуждению. AI-агенты — главный тренд 2025: "
            "автономные системы, решающие задачи без постоянного участия человека. "
            "Мультиагентные архитектуры позволяют масштабировать интеллектуальные "
            "системы. Рынок ИИ оценивается в $250 млрд к 2026 году."
        ),
        "мультиагентные системы": (
            "MAS — архитектурный подход, где задачу решает команда специализированных "
            "агентов. Ключевые паттерны: Supervisor, Handoffs, Map-Reduce. "
            "LangGraph — основной фреймворк для построения MAS на Python. "
            "OpenAI Swarm, CrewAI, AutoGen — альтернативные подходы. "
            "Главная проблема — координация и стоимость множественных вызовов LLM."
        ),
        "квантовые компьютеры": (
            "Google Willow (2024) — 105 кубитов. IBM Heron — 156 кубитов. "
            "Квантовое превосходство подтверждено для узкого класса задач. "
            "Основные области: криптография, моделирование молекул, оптимизация. "
            "До практически полезных квантовых компьютеров — минимум 5-10 лет."
        ),
    }
    for key, value in knowledge.items():
        if key in query.lower():
            return value
    return f"Результаты по запросу '{query}': информация не найдена в демо-базе."


@tool
def format_article(text: str) -> str:
    """Форматирует текст статьи по редакционному стандарту."""
    # Имитация инструмента форматирования
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    formatted = "\n\n".join(paragraphs)
    return f"[Отформатировано: {len(paragraphs)} абзацев]\n\n{formatted}"


@tool
def evaluate_text(text: str) -> str:
    """Оценивает текст по критериям качества: ясность, полнота, стиль."""
    # Имитация инструмента оценки
    word_count = len(text.split())
    has_structure = "##" in text or "\n\n" in text
    score = min(10, word_count // 20 + (3 if has_structure else 0))
    return (
        f"Оценка текста: {score}/10. "
        f"Слов: {word_count}. "
        f"Структура: {'есть' if has_structure else 'отсутствует'}. "
        f"{'Текст достаточного качества.' if score >= 7 else 'Требуется доработка: добавьте деталей и структуры.'}"
    )

## Узел супервайзера

Супервайзер — это LLM в роли главного редактора. Он анализирует текущее состояние (есть ли исследование? есть ли финальный текст?) и решает, кого вызвать следующим: `researcher`, `writer` или `FINISH`. Важно, что супервайзер НЕ вызывает `editor` напрямую — редактор работает автоматически после писателя, внутри цикла рефлексии.

Критически важен fallback при парсинге JSON: LLM не всегда отвечает чистым JSON, поэтому при ошибке парсинга мы ищем имя агента в тексте ответа.

In [5]:
def supervisor_node(state: EditorialState) -> dict:
    """
    Супервайзер: координирует работу редакции.
    Решает, кого вызвать: researcher, writer, editor или FINISH.
    """
    # Собираем контекст о текущем состоянии
    status_parts = []
    if state.get("research"):
        status_parts.append(f"Исследование: есть ({len(state['research'])} символов)")
    else:
        status_parts.append("Исследование: нет")

    if state.get("final_text"):
        status_parts.append(f"Финальный текст: есть ({len(state['final_text'])} символов)")
    elif state.get("draft"):
        status_parts.append(f"Черновик: есть ({len(state['draft'])} символов)")
    else:
        status_parts.append("Текст: нет")

    status = "\n".join(status_parts)

    response = llm.invoke([
        SystemMessage(content="""Ты — главный редактор. Управляешь командой:
- researcher: ищет информацию и факты по теме (вызывай ПЕРВЫМ)
- writer: пишет статью на основе исследования (вызывай ВТОРЫМ, после researcher)
- FINISH: работа завершена (когда есть финальный текст)

ВАЖНО: НЕ вызывай editor напрямую — он работает автоматически после writer.

Верни JSON (и ТОЛЬКО JSON, без markdown-блоков):
{"next": "researcher"} или {"next": "writer"} или {"next": "FINISH"}

Типичный порядок: researcher → writer → FINISH"""),
        HumanMessage(content=f"Задача: {state['task']}\n\nТекущий статус:\n{status}")
    ])

    # Парсинг JSON с fallback
    try:
        parsed = json.loads(response.content.strip())
        next_agent = parsed.get("next", "FINISH")
    except json.JSONDecodeError:
        content = response.content.lower()
        if "researcher" in content:
            next_agent = "researcher"
        elif "writer" in content:
            next_agent = "writer"
        else:
            next_agent = "FINISH"

    print(f"  [Гл. редактор] → {next_agent}")
    return {"next_agent": next_agent}

## Исследователь

Исследователь — первый рабочий агент в цепочке. Он использует инструмент `search_web` для сбора фактов, а затем LLM структурирует найденные данные в ключевые тезисы, цифры и тренды. После завершения управление возвращается супервайзеру.

In [6]:
def researcher_node(state: EditorialState) -> dict:
    """Исследователь: ищет факты по теме с помощью инструмента поиска."""
    raw_data = search_web.invoke({"query": state["task"]})

    response = llm.invoke([
        SystemMessage(content=(
            "Ты — исследователь редакции. Систематизируй найденные факты "
            "для будущей статьи. Выдели ключевые тезисы, цифры, тренды. "
            "Структурируй по пунктам."
        )),
        HumanMessage(content=f"Тема: {state['task']}\n\nНайденные данные:\n{raw_data}")
    ])

    print(f"  [Исследователь] Факты собраны ({len(response.content)} символов)")
    return {"research": response.content}

## Писатель и Редактор — цикл рефлексии

Ядро комбинации — пара Writer и Editor, работающая в петле рефлексии. Писатель создаёт черновик (или дорабатывает его по замечаниям), а редактор оценивает качество с помощью инструмента `evaluate_text`. Если текст не проходит проверку — замечания возвращаются писателю. Если одобрен (APPROVED) или достигнут лимит итераций (2) — управление возвращается супервайзеру.

Супервайзер вызывает `writer` один раз, но внутри может произойти несколько циклов рефлексии. Для супервайзера эта механика прозрачна.

In [7]:
def writer_node(state: EditorialState) -> dict:
    """
    Писатель: создаёт или улучшает статью.
    Если есть feedback от редактора — улучшает с учётом замечаний.
    """
    iteration = state.get("iteration", 0)

    if state.get("feedback") and iteration > 0:
        # Доработка по замечаниям редактора
        prompt = (
            f"Улучши статью с учётом замечаний редактора.\n\n"
            f"Текущая статья:\n{state['draft']}\n\n"
            f"Замечания редактора:\n{state['feedback']}\n\n"
            f"Исходные факты:\n{state['research']}"
        )
    else:
        # Первый черновик
        prompt = (
            f"Напиши статью (5-7 предложений) на основе исследования.\n\n"
            f"Тема: {state['task']}\n\n"
            f"Факты от исследователя:\n{state['research']}"
        )

    # Используем инструмент форматирования
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — автор статей редакции. Пиши увлекательно, "
            "со структурой, опираясь только на предоставленные факты."
        )),
        HumanMessage(content=prompt)
    ])

    formatted = format_article.invoke({"text": response.content})

    print(f"  [Писатель, итерация {iteration + 1}] Черновик: {response.content[:80]}...")
    return {"draft": response.content, "iteration": iteration + 1}

In [8]:
def editor_node(state: EditorialState) -> dict:
    """
    Редактор: оценивает черновик. Использует инструмент evaluate_text.
    Если качество достаточное — одобряет (APPROVED).
    Если нет — возвращает конкретные замечания.
    """
    # Инструментальная оценка
    evaluation = evaluate_text.invoke({"text": state["draft"]})

    response = llm.invoke([
        SystemMessage(content=f"""Ты — строгий редактор. Оцени статью.

Автоматическая оценка: {evaluation}

Критерии:
1. Соответствие теме и фактам
2. Качество стиля и структуры
3. Полнота раскрытия темы

Если статья хорошая — начни ответ со слова APPROVED.
Если нужны правки — дай 2-3 конкретных замечания."""),
        HumanMessage(content=f"Тема: {state['task']}\n\nСтатья:\n{state['draft']}")
    ])

    content = response.content
    iteration = state.get("iteration", 0)

    if "APPROVED" in content.upper():
        print(f"  [Редактор, итерация {iteration}] ОДОБРЕНО!")
        return {
            "feedback": content,
            "final_text": state["draft"],
            "next_agent": "FINISH",
        }

    print(f"  [Редактор, итерация {iteration}] Замечания: {content[:80]}...")

    # Проверяем лимит итераций рефлексии
    if iteration >= 2:
        print(f"  [Редактор] Лимит итераций (2) — принимаем текущую версию")
        return {
            "feedback": content,
            "final_text": state["draft"],
            "next_agent": "FINISH",
        }

    # Не одобрено — возвращаем писателю
    return {
        "feedback": content,
        "next_agent": "writer",
    }

## Маршрутизация и сборка графа

Рефлексия реализована внутри основного графа через условные рёбра. Две функции маршрутизации определяют поток:

- `route_from_supervisor` — направляет к нужному рабочему агенту или завершает граф
- `route_from_editor` — решает, вернуть черновик писателю на доработку или передать управление супервайзеру (если текст одобрен или исчерпан лимит итераций)

Обратите внимание на ключевое ребро `writer → editor` — оно безусловное. Каждый черновик обязательно проходит через редактора. А вот выход из редактора — условный: или обратно к писателю, или наверх к супервайзеру.

In [9]:
def route_from_supervisor(state: EditorialState) -> str:
    """Условное ребро из супервайзера."""
    next_agent = state.get("next_agent", "FINISH")
    if next_agent == "FINISH":
        return END
    return next_agent


def route_from_editor(state: EditorialState) -> str:
    """
    Условное ребро из редактора:
    - APPROVED или лимит → supervisor (который скажет FINISH)
    - Не одобрено → writer (цикл рефлексии)
    """
    next_agent = state.get("next_agent", "FINISH")
    if next_agent == "writer":
        return "writer"
    return "supervisor"


# --- Сборка графа ---
graph = StateGraph(EditorialState)

graph.add_node("supervisor", supervisor_node)
graph.add_node("researcher", researcher_node)
graph.add_node("writer", writer_node)
graph.add_node("editor", editor_node)

graph.add_edge(START, "supervisor")

# Маршрутизация из супервайзера
graph.add_conditional_edges("supervisor", route_from_supervisor, {
    "researcher": "researcher",
    "writer": "writer",
    END: END,
})

# После исследователя — обратно к супервайзеру
graph.add_edge("researcher", "supervisor")

# После писателя — к редактору (цикл рефлексии)
graph.add_edge("writer", "editor")

# Из редактора: если не одобрено → writer, если одобрено → supervisor
graph.add_conditional_edges("editor", route_from_editor, {
    "writer": "writer",
    "supervisor": "supervisor",
})

app = graph.compile(checkpointer=MemorySaver())

## Запуск

Типичный поток выполнения: Supervisor -> Researcher -> Supervisor -> Writer -> Editor -> Writer -> Editor (APPROVED) -> Supervisor -> FINISH. Супервайзер вызвал `writer` один раз, но внутри произошло два цикла рефлексии. Параметр `recursion_limit=30` служит страховкой от бесконечного зацикливания.

In [10]:
config = {
    "configurable": {"thread_id": "editorial-001"},
    "recursion_limit": 30,  # Запас для циклов рефлексии
}

result = app.invoke(
    {
        "task": "Мультиагентные системы: от концепции к production",
        "research": "",
        "draft": "",
        "feedback": "",
        "final_text": "",
        "next_agent": "",
        "iteration": 0,
    },
    config=config,
)

print(f"\n  Итого итераций рефлексии: {result['iteration']}")
print(f"  Исследование: {'есть' if result['research'] else 'нет'}")
print(f"  Финальный текст: {result.get('final_text', 'Нет текста')[:200]}...")
if result.get("final_text"):
    print("  Статус: Статья готова к публикации")

  [Гл. редактор] → researcher


  [Исследователь] Факты собраны (3312 символов)


  [Гл. редактор] → writer


  [Писатель, итерация 1] Черновик: Мультиагентные системы (MAS) — это архитектурный подход, в котором одну задачу р...


  [Редактор, итерация 1] ОДОБРЕНО!


  [Гл. редактор] → FINISH

  Итого итераций рефлексии: 1
  Исследование: есть
  Финальный текст: Мультиагентные системы (MAS) — это архитектурный подход, в котором одну задачу решает не один универсальный LLM-агент, а команда специализированных агентов, разделяющих между собой роли и подзадачи. Т...
  Статус: Статья готова к публикации


## Итоги

В этом примере мы построили комбинацию двух паттернов — **Supervisor** и **Reflection** — в единую архитектуру «виртуальной редакции».

Ключевые принципы:

- **Супервайзер** координирует высокоуровневый поток: исследование -> написание -> завершение
- **Цикл рефлексии** Writer-Editor скрыт от супервайзера — он видит только точку входа (`writer`) и получает готовый результат
- **Лимит итераций** (maxiter=2) гарантирует, что цикл рефлексии не зациклится бесконечно
- **Условные рёбра** реализуют ветвление: из супервайзера — к нужному агенту, из редактора — к писателю или обратно наверх

Этот подход масштабируется: можно добавить больше пар с рефлексией (например, Analyst-Reviewer), вложенные подграфы, параллельные ветки через Map-Reduce. Принцип матрёшки позволяет комбинировать паттерны на разных уровнях абстракции.